In [1]:
# -*- coding: utf-8 -*-

from pathlib import Path
import shutil
import random

import numpy as np
from PIL import Image, ImageOps
import imgaug as ia
import imgaug.augmenters as iaa
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
ia.seed(SEED)

SPLIT_ROOT = Path(
    r"D:\Downloads\Image Preprocessing Project\breast_cancer_split_original"
)

OUTPUT_ROOT = Path(
    r"D:\Downloads\Image Preprocessing Project\breast_cancer_split_train_balanced"
)

CLASSES = ["Non-Cancer", "Cancer"]

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# None معناها: خلي كل كلاس في train يساوي أكبر كلاس موجود في train
# مثال: لو Non-Cancer train = 434 و Cancer train = 87
# Cancer هيتم augment لحد 434
TARGET_TRAIN_COUNT = None

# لو عايز تزود الاتنين لكذا صورة، حط رقم أكبر من أكبر كلاس
# مثال:
# TARGET_TRAIN_COUNT = 1000


seq = iaa.Sequential([
    iaa.Fliplr(0.5),

    # خليت rotation أقل من 45 عشان الصور medical وحساسة
    iaa.Affine(
        rotate=(-15, 15),
        scale=(0.95, 1.05),
        translate_percent={"x": (-0.03, 0.03), "y": (-0.03, 0.03)}
    ),

    iaa.Sometimes(0.35, iaa.GaussianBlur(sigma=(0, 1.0))),
    iaa.Sometimes(0.35, iaa.Dropout(p=(0.01, 0.05))),
    iaa.Sometimes(0.35, iaa.Crop(percent=(0, 0.08), keep_size=True)),
    iaa.Sometimes(0.50, iaa.LinearContrast((0.75, 1.35))),
    iaa.Sometimes(0.50, iaa.Multiply((0.85, 1.15))),
], random_order=True)


def list_images(folder):
    return sorted([
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ])


def safe_copy(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)


def prepare_output_folder():
    if OUTPUT_ROOT.exists():
        shutil.rmtree(OUTPUT_ROOT)

    for split_name in ["train", "val", "test"]:
        for class_name in CLASSES:
            (OUTPUT_ROOT / split_name / class_name).mkdir(parents=True, exist_ok=True)


def copy_split(split_name):
    for class_name in CLASSES:
        src_folder = SPLIT_ROOT / split_name / class_name
        dst_folder = OUTPUT_ROOT / split_name / class_name

        if not src_folder.exists():
            raise FileNotFoundError(f"Folder not found: {src_folder}")

        files = list_images(src_folder)

        for src in files:
            dst = dst_folder / src.name
            safe_copy(src, dst)


def load_image_as_array(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)

    # نخلي الصورة زي ما هي لو Grayscale أو RGB
    if img.mode not in ["L", "RGB"]:
        img = img.convert("RGB")

    arr = np.array(img)
    return arr, img.mode


def save_array_as_image(arr, output_path, mode):
    arr = np.clip(arr, 0, 255).astype(np.uint8)

    if arr.ndim == 2:
        img = Image.fromarray(arr, mode="L")
    else:
        img = Image.fromarray(arr)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    img.save(output_path)


def augment_class_until_target(class_name, target_count):
    train_src_folder = SPLIT_ROOT / "train" / class_name
    train_dst_folder = OUTPUT_ROOT / "train" / class_name

    original_files = list_images(train_src_folder)
    current_files = list_images(train_dst_folder)

    current_count = len(current_files)
    needed = target_count - current_count

    if needed <= 0:
        print(f"{class_name}: already has {current_count} images. No augmentation needed.")
        return

    print(f"{class_name}: augmenting {needed} images to reach {target_count}.")

    for aug_index in tqdm(range(needed), desc=f"Augmenting {class_name}"):
        src = random.choice(original_files)

        arr, mode = load_image_as_array(src)
        aug_arr = seq(image=arr)

        output_name = f"{src.stem}_aug_{aug_index + 1:05d}{src.suffix}"
        output_path = train_dst_folder / output_name

        save_array_as_image(aug_arr, output_path, mode)


def print_counts():
    print("\nFinal dataset counts:\n")

    for split_name in ["train", "val", "test"]:
        print(split_name.upper())
        for class_name in CLASSES:
            folder = OUTPUT_ROOT / split_name / class_name
            count = len(list_images(folder))
            print(f"  {class_name}: {count}")
        print()


def main():
    prepare_output_folder()

    # Copy all original split files first
    copy_split("train")
    copy_split("val")
    copy_split("test")

    train_counts = {}

    for class_name in CLASSES:
        folder = OUTPUT_ROOT / "train" / class_name
        train_counts[class_name] = len(list_images(folder))

    if TARGET_TRAIN_COUNT is None:
        target_count = max(train_counts.values())
    else:
        target_count = TARGET_TRAIN_COUNT

    print("Original train counts:")
    for class_name, count in train_counts.items():
        print(f"  {class_name}: {count}")

    print(f"\nTarget train count per class: {target_count}\n")

    for class_name in CLASSES:
        augment_class_until_target(class_name, target_count)

    print_counts()


if __name__ == "__main__":
    main()

Original train counts:
  Non-Cancer: 434
  Cancer: 87

Target train count per class: 434

Non-Cancer: already has 434 images. No augmentation needed.
Cancer: augmenting 347 images to reach 434.


Augmenting Cancer: 100%|██████████| 347/347 [00:45<00:00,  7.70it/s]


Final dataset counts:

TRAIN
  Non-Cancer: 434
  Cancer: 434

VAL
  Non-Cancer: 93
  Cancer: 19

TEST
  Non-Cancer: 93
  Cancer: 19

